# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes
print(f"Dataset loaded: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

First, list all available record sets and their `@id` fields, then enumerate available fields/columns for one record set.

In [ ]:
# List all available record sets and their @id fields
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets were found in this dataset.")
else:
    print("Available Record Sets and their @id fields:")
    for rs in record_sets:
        print(f"- Name: {rs.name}")
        print(f"  @id: {rs.id}")
        print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
        print(f"  Number of fields: {len(rs.fields)}")

    # Example: Inspect fields for first record set
    first_record_set = record_sets[0]
    print("\nFields in first record set (by @id):")
    for field in first_record_set.fields:
        print(f"- {field.name}: {field.id} (type: {field.data_type if hasattr(field, 'data_type') else ''})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set by @id
import itertools

# Gather all record set @id fields
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records):
        df = pd.DataFrame(records)
        # Ensure column order consistent
        rs = next(rs for rs in dataset.metadata.record_sets if rs.id == record_set_id)
        field_ids = [field.id for field in rs.fields]
        cols = [c for c in field_ids if c in df.columns]
        dataframes[record_set_id] = df[cols] if cols else df
        print(f"Loaded record set {record_set_id} with shape {df.shape}")
    else:
        print(f"No records found in record set {record_set_id}")

# Show columns for the main record set
if record_set_ids:
    main_rs = record_set_ids[0]
    print(f"\nFields (@id) in main record set: {dataframes[main_rs].columns.tolist()}\n")
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis
# Choose first numeric field found in schema (change as appropriate)
rs = dataset.metadata.record_sets[0]
numeric_field_id = None
for field in rs.fields:
    # Simple heuristic to find a float/int field
    if hasattr(field, 'data_type') and field.data_type in ['schema:Float', 'schema:Integer', 'Float', 'Integer']:
        numeric_field_id = field.id
        break

if numeric_field_id is None:
    raise ValueError("No numeric field found for EDA.")
print(f"Using numeric field for analysis: {numeric_field_id}")

main_rs = dataset.metadata.record_sets[0].id
df = dataframes[main_rs]

# Attempt conversion to numeric in case it's parsed as string
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id]).all() else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optional grouping by a categorical field (choose first string field if available)
group_field = None
for field in rs.fields:
    # Simple heuristic for choosing group: string columns or those with < 20 unique values
    if field.id != numeric_field_id and df[field.id].dtype == object:
        num_uniques = df[field.id].nunique()
        if num_uniques < 20:
            group_field = field.id
            break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field:
    plt.figure(figsize=(12, 5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field} (filtered)")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded mass spectrometry analysis dataset using `mlcroissant` via its Croissant schema.
- Explored record sets and inspected their fields using each schema element's `@id`.
- Performed EDA on one numeric field, including filtering, normalization, and optional grouping by a categorical field.
- Visualized the selected numeric field distribution and grouped statistics.

**Next steps:** Consider more domain-specific analyses, data cleaning steps, or combining these features with experimental or external labels for advanced modeling.